Scenario
“A hospital uses different employees (agents) to handle patient care in sequence.”

AGENT 1: Intake Agent (Planner)
- Collects patient symptoms and history
- Decides which steps are needed (tests, consultations, etc.)

AGENT 2: Diagnostic Agent
- Orders lab tests or scans
- Interprets results and identifies possible conditions

AGENT 3: Treatment Agent
- Suggests treatment options (medication, therapy, surgery)
- Considers patient preferences and medical guidelines

AGENT 4: Decision Agent
- Reviews all inputs (history, diagnostics, treatment options)
- Provides the final recommendation to the patient

In [ ]:
!pip install groq

In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get('newapi')

In [ ]:
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('newapi'))

api_key = userdata.get('newapi')

if not api_key:
    raise ValueError("API key not found. Please add it in Colab Secrets.")

client = Groq(api_key=api_key)

In [ ]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # ✅ updated model
        messages=[
            {"role": "system", "content": "You are a precise medical AI assistant. Always return valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


# AGENT 1: INTAKE AGENT
def intake_agent(user_query):
    print("\n[Intake Agent] Extracting symptoms and history...")

    prompt = f"""
    You are a medical intake assistant.

    Extract:
    - Symptoms (list)
    - Medical history (if any)

    Input: {user_query}

    Output JSON:
    {{
      "symptoms": [],
      "history": ""
    }}
    """

    return call_llm(prompt)


# AGENT 2: DIAGNOSTIC AGENT
def diagnostic_agent(intake_data):
    print("\n[Diagnostic Agent] Diagnosing condition...")

    prompt = f"""
    You are a medical diagnostic AI.

    Based on patient data:
    {intake_data}

    Provide possible conditions with risk (1-10).

    Output JSON:
    [
      {{"condition": "", "risk": 0}}
    ]
    """

    return call_llm(prompt)


# AGENT 3: TREATMENT AGENT
def treatment_agent(diagnosis):
    print("\n[Treatment Agent] Suggesting treatment...")

    prompt = f"""
    You are a treatment recommendation AI.

    Based on diagnosis:
    {diagnosis}

    Suggest:
    - medication (true/false)
    - therapy (true/false)
    - surgery (true/false)
    - advice

    Output JSON:
    {{
      "medication": true,
      "therapy": false,
      "surgery": false,
      "advice": ""
    }}
    """

    return call_llm(prompt)


# AGENT 4: DECISION AGENT
def decision_agent(intake_data, diagnosis, treatment):
    print("\n[Decision Agent] Final decision...")

    prompt = f"""
    You are a senior medical decision AI.

    Review:
    Intake: {intake_data}
    Diagnosis: {diagnosis}
    Treatment: {treatment}

    Provide final recommendation summary.

    Output:
    {{
      "final_decision": ""
    }}
    """

    return call_llm(prompt)


# MAIN SYSTEM
def hospital_multi_agent(user_query):
    print("User Query:", user_query)

    intake = intake_agent(user_query)
    diagnosis = diagnostic_agent(intake)
    treatment = treatment_agent(diagnosis)
    decision = decision_agent(intake, diagnosis, treatment)

    return decision

# RUN
response = hospital_multi_agent("Patient has chest pain, dizziness and high BP")

print("\n===== FINAL OUTPUT =====")
print(response)

User Query: Patient has chest pain, dizziness and high BP

[Intake Agent] Extracting symptoms and history...

[Diagnostic Agent] Diagnosing condition...

[Treatment Agent] Suggesting treatment...

[Decision Agent] Final decision...

===== FINAL OUTPUT =====
```json
{
  "final_decision": "The patient should be treated with medication to manage blood pressure, cholesterol, and heart rhythm, and undergo therapy sessions to address anxiety disorder and provide lifestyle counseling. Close monitoring for cardiovascular complications is crucial, and surgery may be considered in the future if necessary."
}
```
